In [ ]:
# ============================================================================
# INPUT CHECK - runs first, before the stage below it.
#
# Every stage resolves its inputs with glob("/kaggle/input/**/<name>") and takes
# the FIRST hit alphabetically. A file attached twice is therefore not an error,
# it is a coin toss - and the order is worse than random: a folder named
# stage1-data-pipeline-OLD sorts BEFORE stage1-data-pipeline, so the copy you
# meant to retire is the one that wins, silently.
#
# This prints what will actually be read, and stops the run if anything is
# attached twice. Everything lives inside one function so it cannot collide
# with a name the stage below uses.
# ============================================================================
def _check_kaggle_inputs():
    import glob, os
    from datetime import datetime

    WANTED = {
        "unified.parquet":                     "Stage 1 corpus        (Stages 2, 4)",
        "splits.json":                         "Stage 1 splits        (Stage 2)",
        "predictions_finetuned.parquet":       "Stage 2 encoder store (Stage 2 resume, 4, 5)",
        "predictions_llm.parquet":             "Stage 3 LLM binary    (Stages 4, 5)",
        "predictions_subtype.parquet":         "Stage 3b raw subtype  (Stage 3b-repair)",
        "predictions_subtype_repaired.parquet": "Stage 3b repaired    (Stages 4, 5)",
        "tab9_evaluability.csv":               "Stage 4 gate          (Stage 5)",
    }
    # Files THIS stage actually reads. A bad version of one of these is a
    # hard stop; anything else in WANTED is checked for information only.
    CONSUMED = {"unified.parquet", "splits.json", "predictions_finetuned.parquet"}

    print("=" * 78)
    print("ATTACHED INPUTS")
    print("=" * 78)
    problems = []
    for fname, used_by in WANTED.items():
        hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
        print(f"\n{fname}   <- {used_by}")
        if not hits:
            print("   (not attached)")
            continue
        for i, h in enumerate(hits):
            when = datetime.fromtimestamp(os.path.getmtime(h)).strftime("%Y-%m-%d %H:%M")
            mark = "  >> THIS ONE WILL BE USED" if i == 0 else "     ignored"
            print(f"   {h}\n      {os.path.getsize(h):>12,} bytes   {when}{mark}")
        if len(hits) > 1:
            problems.append(f"DUPLICATE: {fname} is attached {len(hits)} times")

    print("\n" + "=" * 78)
    print("ROW COUNTS OF WHAT WILL ACTUALLY BE READ")
    print("=" * 78)
    # A file of the right NAME can still be the wrong VERSION. The row count is
    # what tells them apart. The encoder store has two legitimate sizes
    # depending on where you are in the chain, so it is checked against both.
    EXPECTED = {
        "unified.parquet":                     ({1412}, "1,412"),
        "predictions_finetuned.parquet":       ({16900, 24280},
                                                "16,900 before Stage 2 / 24,280 after it"),
        "predictions_llm.parquet":             ({35049}, "35,049"),
        "predictions_subtype_repaired.parquet": ({16786}, "16,786"),
    }
    try:
        import pandas as pd
        for fname, (ok_values, note) in EXPECTED.items():
            hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
            if not hits:
                continue
            n = len(pd.read_parquet(hits[0]))
            # The expected values are the counts of the COMMITTED REFERENCE
            # run. A LARGER prediction store is legitimate after an API top-up
            # (the harnesses resume and add previously rate-limited items); a
            # SMALLER one is an old or partial store and must not be used.
            # unified.parquet must match exactly: requirement ids are
            # positional, so a corpus of any other size silently re-points
            # every stored prediction.
            if n in ok_values:
                flag = "OK"
            elif fname == "unified.parquet":
                flag = "<-- CORPUS SIZE CHANGED: ids are positional, do NOT run"
            elif n > max(ok_values):
                flag = ("larger than the committed reference (expected after an "
                        "API top-up; verify provenance, then update EXPECTED here)")
            else:
                flag = ("<-- SMALLER than the committed reference "
                        "(old/partial store), do not run")
            print(f"  {fname:38s} {n:>7,} rows   expected {note}   {flag}")
            # A '<--' flag on a file THIS stage actually consumes is a hard
            # stop: the old check printed 'do not run' and then ran anyway,
            # which is how hours of compute get spent against a wrong input.
            if flag.startswith("<--") and fname in CONSUMED:
                problems.append(f"BAD INPUT: {fname} - {flag.lstrip('<- ')}")
            if fname == "predictions_finetuned.parquet":
                which = ("the OLD store (correct INPUT for Stage 2 itself)" if n == 16900
                         else "the NEW store (required by Stages 4 and 5)" if n == 24280
                         else "neither size this chain produces")
                print(f"      -> this is {which}")
    except Exception as e:
        print("  (could not read:", e, ")")

    sp = sorted(glob.glob("/kaggle/input/**/splits.json", recursive=True))
    if sp:
        import json
        fams = json.load(open(sp[0]))
        n_folds = sum(len(v) for v in fams.values())
        verdict = "OK" if len(fams) == 14 else "<-- OLD Stage 1 output, re-run Stage 1 first"
        print(f"\n  splits.json: {len(fams)} families, {n_folds} folds   {verdict}")
        if verdict.startswith("<--") and "splits.json" in CONSUMED:
            problems.append("BAD INPUT: splits.json - OLD Stage 1 output "
                            f"({len(fams)} families, expected 14)")

    print("\n" + "=" * 78)
    if problems:
        for p in problems:
            print(f"  {p}")
        raise SystemExit("Fix the problems above (detach duplicates / attach "
                         "the right versions), then run again. "
                         "The stage below did NOT run.")
    print("No duplicates. Running the stage now.")
    print("=" * 78 + "\n")


_check_kaggle_inputs()


# =============================================================================
# STAGE 2 - FINE-TUNED TRANSFORMER BASELINES
#
# The supervised anchor the prompted LLMs are compared against. No novelty is
# claimed here; the point is a fair, identical protocol on the SAME splits
# Stage 1 produced.
#
# WHAT CHANGED, and why:
#
#   1. SUPERSEDES notebooks 02, 02.1 and stage2-merged. Those three are now
#      redundant and should be deleted or archived.
#
#   2. RUNS THE IN-DOMAIN REFERENCE ARMS. Previously only the out-of-
#      distribution folds were run, so RQ2's "generalisation gap" had no
#      in-distribution number to subtract from. Every split family Stage 1
#      emits is now executed.
#
#   3. MULTI-CLASS SUB-TYPE BASELINE. There was previously no fine-tuned
#      comparator for the NFR sub-type task at all, so that chapter compared
#      LLMs against nothing but a majority-class baseline. The runner is now
#      label-set driven and handles all-11 / top-6 / top-4.
#
#   4. CLASS WEIGHTING IS EXPLICIT AND AUDITED. `class_weighting` is a stored
#      column on every prediction row ("balanced" or "none"), never inferred
#      from the model tag. The unweighted control now runs on the in-domain
#      arms as well as the cross-dataset arms, so the claim about imbalance is
#      supported on both sides of the gap rather than only one.
#
#   5. HONEST INSTRUMENTATION. Real per-item inference latency and real
#      tokenizer input-token counts are recorded for every prediction, so
#      Stage 5 never has to impute or emit NaN. `completion_tokens` is 0 by
#      construction for an encoder classifier (there is no generated text) and
#      `cost_basis` records that the cost model is GPU seconds, not API tokens.
#
#   6. LABELS ARE ALWAYS STORED AS STRINGS. The earlier string/int mismatch
#      between stores (which silently broke Stage 4) cannot recur.
#
# OUTPUTS -> /kaggle/working/
#      predictions_finetuned.parquet (+ .csv)
#      summary_finetuned.csv
#      class_weighting_effect.csv
#      stage2_manifest.json
#
# KAGGLE: Accelerator = GPU T4. Internet = ON (model download).
# INPUT  : + Add Input -> Notebook Output -> Stage 1
# RUNTIME: roughly 3-4 h on a T4 for the full plan. Set quick_mode=True first.
# =============================================================================

import glob
import json
import logging
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
# Seeding the RNGs is not by itself reproducibility on a GPU. The cuDNN flags
# below cover only cuDNN kernels (convolutions), which BERT/RoBERTa barely
# use; use_deterministic_algorithms is what steers the matmul/scatter kernels
# these models actually run through to deterministic implementations where
# they exist (warn_only, so an op with no deterministic path warns instead of
# crashing the run). Even together these make re-runs REPRODUCIBLE IN
# EXPECTATION, not bitwise-guaranteed - state it that way in the write-up
# rather than claiming exact replay.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
log = logging.getLogger("stage2")

CONFIG = {
    # Auto-discovered under /kaggle/input; set explicitly to override.
    "data_dir": None,
    "out_dir": "/kaggle/working",

    "models": ["bert-base-uncased", "roberta-base"],

    # Which split families to run.
    #
    # None = EVERY family Stage 1 emits. This file's own header claims "Every
    # split family Stage 1 emits is now executed", and with a hand-written list
    # that was true only by coincidence: Stage 1 gained six cross-project
    # families and this list would have silently ignored all of them, leaving
    # RQ2 unanswered for the sub-type task while the header said otherwise.
    # Deriving the list makes the claim true by construction.
    #
    # Set an explicit list only to run a subset deliberately, e.g. for a
    # rehearsal. Anything named here must exist in splits.json or the run stops.
    "families": None,
    # Families to skip even when running everything. Use for a family that is
    # deliberately out of scope, and say why here so it is not a mystery later.
    "skip_families": [],

    # ---- CLASS WEIGHTING (see the audit note at the bottom of this file) ----
    # Primary condition: inverse-frequency ("balanced") weighting everywhere.
    "weighting_primary": "balanced",
    # Unweighted control, for the imbalance claim. Runs on both an in-domain
    # family and a cross-dataset family so the effect is measured on BOTH
    # sides of the generalisation gap, not just the transfer side.
    "run_unweighted_control": True,
    "control_families": ["cross_dataset", "in_domain_security_promise"],

    "max_len": 128,
    "epochs": 3,

    # ---- EPOCH OVERRIDE, KEYED ON TASK ------------------------------------
    # The sub-type families are much smaller than the binary ones: all-11 trains
    # on ~419 examples, which at batch 16 for 3 epochs is only ~80 optimiser
    # steps for an eleven-way problem. That is not enough, and an under-trained
    # baseline would invite the objection that the sub-type comparison was
    # unfair to the encoder. These families therefore train longer.
    #
    # Report this in the thesis as a deliberate, stated deviation: hyperparameters
    # are uniform WITHIN each task, and the deviation FAVOURS the baseline, not
    # the LLMs. An adjustment that strengthens your comparator is defensible; one
    # that weakens it would not be.
    #
    # THE OVERRIDE IS KEYED ON TASK, NOT FAMILY, AND THAT IS LOAD-BEARING. It
    # used to name three families as literals - in_domain_subtype_{top4,top6,all}
    # - back when those were the only sub-type families in existence. Stage 1
    # now also emits cross_project_subtype_*_promise, which is the same task on
    # training sets of the same size (276-423 examples), and under a family-keyed
    # override those would have silently trained for 3 epochs against the
    # in-domain arm's 10. The headline sub-type number for RQ2 is
    #     gap = macro_F1(in_domain) - macro_F1(cross_project)
    # so that difference would have subtracted a 10-epoch model from a 3-epoch
    # one and reported the result as a generalisation gap. Epoch count and
    # evaluation regime would have been perfectly confounded, in the one figure
    # the study calls its primary contribution. Keying on task makes the two
    # arms comparable by construction, and the MIXED-epoch guard below cannot
    # catch this class of error because it only looks WITHIN a result group.
    "task_epochs": {
        "subtype_top4": 10,
        "subtype_top6": 10,
        "subtype_all": 10,
    },
    # Per-family escape hatch, for a deliberate one-off that a task-wide rule
    # should not express. Takes precedence over task_epochs. Anything put here
    # re-introduces the confound above unless BOTH arms of the same task are
    # given the same value, so state the reason alongside the entry.
    "family_epochs": {},

    # Families listed here are DELETED from any existing prediction store and
    # retrained from scratch. Use this after changing hyperparameters for a
    # family: without it the resume logic would keep the old rows and you would
    # silently end up with a store where some folds used different settings.
    # Set back to [] once the re-run is complete.
    # Emptied now that the 10-epoch sub-type re-run has completed. Leaving the
    # families listed makes EVERY subsequent run purge 15 finished folds x 2
    # models = 30 runs (2,620 rows, ~42 min of T4 training time) and retrain
    # them under the settings they already used. Re-populate only when a
    # hyperparameter actually changes, then empty it again once the re-run is
    # committed.
    "force_rerun_families": [],

    "lr": 2e-5,
    "train_bs": 16,
    # 1, deliberately. Stage 2 used to time a batch of 32 and write dt/32 to
    # every row, while Stages 3 and 3b time one generate() call per item. Stage 5
    # then priced both as the same quantity, stated in its provenance and figure
    # captions that "latency was measured under sequential unbatched inference",
    # and divided the already-amortised encoder figure by BATCH_SPEEDUPS
    # [1, 8, 32] again in its batching sensitivity - a speed-up counted twice.
    # Predictions are unaffected: make_loader pads every sequence to max_length,
    # so batch size changes throughput, not output. The cost is roughly six
    # minutes of extra GPU time over the whole plan.
    "eval_bs": 1,

    # Imputed hardware rental used to price local compute, identical to the
    # value Stages 3, 3b and 5 use. Stored per row as cost_usd so the encoder
    # no longer arrives at the cost analysis with a missing column.
    # AWS EC2 g4dn.xlarge (1x T4), on-demand, us-east-1, verified 2026-08-06.
    "gpu_usd_per_hour": 0.5260,

    # Checkpoint interval, in completed folds. The previous code rewrote the
    # ENTIRE store (csv + parquet) after every fold, which is quadratic: by the
    # last of the 228 planned runs it was rewriting ~24k rows to two formats to
    # append a few hundred. Every fold is still durable within this many
    # folds, and the store is always written once more at the end.
    "checkpoint_every": 5,

    # Plumbing check: real model, 1 epoch, 2 folds per family, ~5 minutes.
    "quick_mode": False,
    "quick_models": ["bert-base-uncased"],
    "quick_max_folds": 2,
    "quick_epochs": 1,
}

# Canonical prediction schema, shared byte-for-byte with Stage 3 and Stage 3b.
# CANONICAL PREDICTION SCHEMA - shared verbatim by Stages 2, 3 and 3b.
#
# This list previously differed between the stages: Stage 2 carried
# `train_epochs` while Stages 3/3b carried `cost_usd`, `quantization` and
# `timestamp`, even though all three headers claimed the schema was identical.
# Stage 4 and Stage 5 survived only because pd.concat(sort=False) silently
# filled the gaps with NaN - which meant every encoder row reached the cost
# analysis with cost_usd = 0 and was rescued by a special case. One list, three
# stages, no special cases.
#
#   train_epochs  0 for a prompted model (nothing is trained)
#   cost_usd      imputed GPU rental for local compute, billed tokens for an API
#   quantization  "none_fp32" for an encoder, "4bit_nf4" for a quantised LLM
STORE_COLS = ["model_tag", "model_type", "approach", "class_weighting",
              "task", "dataset", "eval_regime", "family", "fold",
              "split", "prompt_id", "shot_k",
              "id", "y_true", "y_pred", "parse_ok",
              "prompt_tokens", "completion_tokens", "latency_s",
              "train_time_s", "train_epochs", "price_in_per_mtok",
              "price_out_per_mtok", "cost_usd", "cost_basis", "quantization",
              "timestamp", "raw_output"]

# FIXED LABEL SETS, identical to Stage 4 and Stage 5.
#
# summarise() used to derive the label set from the values it observed
# (`sorted(set(y_true) | set(y_pred))`). For an eleven-way sub-type fold where a
# rare class happens not to appear, that divides the macro average by a smaller
# K and reports a HIGHER macro-F1 than Stage 4 reports for the same cell. Stage
# 5 documents this exact hazard in its [FIX-6] note; the fix belongs here too,
# or summary_finetuned.csv and tab1 disagree in the write-up.
CATEGORIES_ALL = ["availability", "fault_tolerance", "legal", "look_and_feel",
                  "maintainability", "operational", "performance", "portability",
                  "scalability", "security", "usability"]
TOP4 = ["security", "usability", "operational", "performance"]
TOP6 = TOP4 + ["look_and_feel", "availability"]
LABELSETS = {
    "fr_nfr": ["FR", "NFR"],
    "security": ["non-security", "security"],
    "subtype_all": CATEGORIES_ALL,
    "subtype_top6": TOP6,
    "subtype_top4": TOP4,
}
BINARY_LABELSETS = {k: LABELSETS[k] for k in ("fr_nfr", "security")}


# =============================================================================
# io helpers  (parquet where available, csv always -- never lose a run)
# =============================================================================
def find_data_dir(explicit=None):
    if explicit and (Path(explicit) / "splits.json").exists():
        return str(explicit)
    hits = sorted(glob.glob("/kaggle/input/**/unified.parquet", recursive=True))
    hits += sorted(glob.glob("/kaggle/input/**/unified.csv", recursive=True))
    if not hits:
        raise FileNotFoundError(
            "Stage 1 output not found under /kaggle/input.\n"
            "Fix: + Add Input -> Notebook Output -> select the Stage 1 notebook.")
    return str(Path(hits[0]).parent)


def read_table(path_no_ext):
    """Resume path. keep_default_na=False matters: a stored string such as
    "n/a" would otherwise be read back as NaN and the resume logic would
    re-run work it had already completed (or drop it)."""
    for ext in (".parquet", ".csv"):
        p = f"{path_no_ext}{ext}"
        if os.path.exists(p):
            try:
                if ext == ".parquet":
                    return pd.read_parquet(p)
                return pd.read_csv(p, keep_default_na=False, na_values=[""])
            except Exception as e:
                log.warning("could not read %s (%s)", p, e)
    return None


def write_table(df, path_no_ext):
    df.to_csv(f"{path_no_ext}.csv", index=False)
    try:
        df.to_parquet(f"{path_no_ext}.parquet", index=False)
    except Exception as e:
        log.warning("parquet write skipped (%s); csv written.", e)


# =============================================================================
# metrics  (macro AND weighted -- the Literature Review promises both)
# =============================================================================
def compute_metrics(y_true, y_pred, labelset):
    y_true, y_pred = list(y_true), list(y_pred)
    return {
        "n": len(y_true),
        "macro_f1": f1_score(y_true, y_pred, labels=labelset, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labelset, average="weighted", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, labels=labelset, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=labelset, average="macro", zero_division=0),
    }


# =============================================================================
# training / prediction
# =============================================================================
def make_loader(texts, labels, tokenizer, max_len, bs, shuffle):
    enc = tokenizer(list(texts), truncation=True, max_length=max_len,
                    padding="max_length", return_tensors="pt")
    y = torch.tensor(labels, dtype=torch.long)
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], y)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)


def real_token_counts(texts, tokenizer, max_len):
    """Actual tokenizer input length per requirement -- not padded length, and
    never a placeholder. Feeds the RQ3 cost model."""
    enc = tokenizer(list(texts), truncation=True, max_length=max_len)
    return [len(x) for x in enc["input_ids"]]


def finetune_and_predict(model_name, train_df, test_df, label_col, labelset,
                         hp, weighting):
    """Returns (pred_df, train_time_s). pred_df carries per-item latency and
    real input-token counts."""
    lab2id = {lab: i for i, lab in enumerate(labelset)}
    id2lab = {i: lab for lab, i in lab2id.items()}

    train_df = train_df[train_df[label_col].astype(str).isin(labelset)]
    test_df = test_df[test_df[label_col].astype(str).isin(labelset)]
    ytr = train_df[label_col].astype(str).map(lab2id).to_numpy()
    yte = test_df[label_col].astype(str).map(lab2id).to_numpy()

    # ---- class weighting: explicit, logged, and stored on every row --------
    class_weights = None
    if weighting == "balanced":
        present = np.unique(ytr)
        w = compute_class_weight("balanced", classes=present, y=ytr)
        # classes absent from this fold get weight 1.0 (they contribute no loss)
        full = np.ones(len(labelset), dtype=float)
        for c, wt in zip(present, w):
            full[c] = wt
        class_weights = torch.tensor(full, dtype=torch.float).to(DEVICE)
        log.info("  class_weighting=balanced %s",
                 {id2lab[i]: round(float(v), 3) for i, v in enumerate(full)})
    else:
        log.info("  class_weighting=none (unweighted control)")

    tok = AutoTokenizer.from_pretrained(model_name)
    torch.manual_seed(SEED)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(labelset)).to(DEVICE)

    tr_loader = make_loader(train_df["text"], ytr.tolist(), tok,
                            hp["max_len"], hp["train_bs"], True)
    optim = torch.optim.AdamW(model.parameters(), lr=hp["lr"])
    sched = get_linear_schedule_with_warmup(optim, 0, len(tr_loader) * hp["epochs"])

    # ---- train ----
    t0 = time.time()
    model.train()
    for _ in range(hp["epochs"]):
        for ids, mask, y in tr_loader:
            ids, mask, y = ids.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
            optim.zero_grad()
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits, y, weight=class_weights)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step(); sched.step()
    train_time = time.time() - t0

    # ---- predict, timing each BATCH so per-item latency is real ------------
    te_loader = make_loader(test_df["text"], yte.tolist(), tok,
                            hp["max_len"], hp["eval_bs"], False)
    model.eval()
    preds, per_item_latency = [], []
    with torch.no_grad():
        for ids, mask, _ in te_loader:
            ids, mask = ids.to(DEVICE), mask.to(DEVICE)
            t1 = time.time()
            logits = model(input_ids=ids, attention_mask=mask).logits
            if DEVICE == "cuda":
                torch.cuda.synchronize()      # otherwise the timing is a lie
            dt = time.time() - t1
            b = ids.shape[0]
            preds.extend(logits.argmax(-1).cpu().tolist())
            per_item_latency.extend([dt / b] * b)

    tok_in = real_token_counts(test_df["text"], tok, hp["max_len"])

    del model, optim, sched, tr_loader, te_loader
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    pred_df = pd.DataFrame({
        "id": test_df["id"].values,
        "y_true": [id2lab[i] for i in yte],
        "y_pred": [id2lab[i] for i in preds],
        "prompt_tokens": tok_in,
        "completion_tokens": 0,        # encoder classifier: nothing is generated
        "latency_s": [round(x, 6) for x in per_item_latency],
    })
    return pred_df, train_time


# =============================================================================
# run plan
# =============================================================================
def build_plan(splits, cfg):
    """(model_tag, base_model, weighting, family, entry) in execution order."""
    q = cfg["quick_mode"]
    models = cfg["quick_models"] if q else cfg["models"]
    plan = []

    for m in models:
        for fam in cfg["families"]:
            entries = splits.get(fam, [])
            if q:
                entries = sorted(entries, key=lambda e: -len(e["test_ids"]))[:cfg["quick_max_folds"]]
            for e in entries:
                plan.append((f"{m}-weighted", m, cfg["weighting_primary"], fam, e))

    if cfg["run_unweighted_control"] and not q:
        for m in models:
            for fam in cfg["control_families"]:
                for e in splits.get(fam, []):
                    plan.append((f"{m}-unweighted", m, "none", fam, e))
    return plan


# Stage 1 emits TWO cross-project partitions of the fr_nfr task over the same
# 968 PROMISE requirements: leave_project_out (47 LOPO folds, built for the
# per-project variance plot and meant to be scored pooled) and
# cross_project_fr_nfr_promise (5 grouped-CV folds, built as the reportable
# transfer arm). Both carry eval_regime="cross_project" and task="fr_nfr".
#
# Stage 4 groups on ["model","model_tag","tier","task","dataset","eval_regime"]
# and Stage 5 on the same six keys. Neither reads `family` - the word does not
# appear in either file. So with both families in the store they collapse into
# ONE cell in which every requirement is scored twice: n=1936 for a 968-item
# corpus, a headline transfer number that is a 50/50 blend of two different
# experiments, and a bootstrap CI ~30% narrower purely because each item is
# resampled twice. tab4 and tab11 call drop_duplicates("id") and would keep
# whichever family was written first, so those two tables would silently
# describe a different experiment than tab1/tab2/tab3.
#
# The family column cannot fix this while nothing downstream reads it. Putting
# the distinction in eval_regime puts it in the key that IS read: the pooled
# variance family gets its own regime, so it stays out of the transfer tables
# (TRANSFER lists "cross_project", not this) while Stage 4's per-project table
# still finds it by prefix.
POOLED_VARIANCE_FAMILIES = {"leave_project_out": "cross_project_lopo"}

# (family, fold) -> training-set size. Kept beside the store rather than
# inside it because pred_df[STORE_COLS] drops any column not in the shared
# 28-column schema, so a row-level field would have been silently discarded
# and mean_train_n would have been NaN in every row of the summary - a fix
# that looks applied and does nothing. Filled for EVERY fold of EVERY family
# straight from splits.json at the start of run(): fold sizes are a
# deterministic property of the splits, so rows resumed from an earlier run
# get the same correct size as freshly-trained ones. (It used to be filled
# only inside the training loop, so a fully-resumed re-run regenerated the
# summaries with mean_train_n = NaN and generalisation-gap size_matched
# silently flipping to False.)
FOLD_TRAIN_N = {}


def regime_of(family, entry):
    return POOLED_VARIANCE_FAMILIES.get(family, entry["eval_regime"])


def dataset_of(uni_by_id, ids):
    vals = uni_by_id.loc[list(ids), "source_dataset"].unique()
    return vals[0] if len(vals) == 1 else "mixed"


def epochs_for(family, task, cfg):
    """Epochs for one split family. quick_mode overrides everything; otherwise a
    family-specific override wins over a task-wide one, which wins over the
    global default. Resolving by TASK is what keeps the two arms of a
    generalisation gap on the same training budget - see task_epochs above."""
    if cfg["quick_mode"]:
        return cfg["quick_epochs"]
    if family in cfg.get("family_epochs", {}):
        return int(cfg["family_epochs"][family])
    return int(cfg.get("task_epochs", {}).get(task, cfg["epochs"]))


def purge_families(existing, cfg):
    """Drop rows belonging to families listed in force_rerun_families.

    Without this, the resume logic keys on (model_tag, fold) and would happily
    keep rows trained under the OLD hyperparameters, leaving a store in which
    some folds used 3 epochs and others 10. That inconsistency is invisible in
    the summary table and would be very hard to detect later."""
    force = list(cfg.get("force_rerun_families", []))
    if existing is None or not force:
        return existing, 0
    if "family" not in existing.columns:
        log.warning("Existing store has no 'family' column; cannot purge "
                    "selectively. Delete the store manually if hyperparameters "
                    "changed.")
        return existing, 0
    mask = existing["family"].isin(force)
    n = int(mask.sum())
    if n:
        by_fam = existing.loc[mask, "family"].value_counts().to_dict()
        log.warning("PURGING %d rows from %s so they retrain under the new "
                    "settings. Rows from all other families are kept.", n, by_fam)
    return existing.loc[~mask].reset_index(drop=True), n


def seed_store_from_inputs(cfg):
    """Copy a previous prediction store from /kaggle/input into out_dir.

    /kaggle/working is EMPTY at the start of every Kaggle session, while a
    previous run's store lives in that notebook's committed output (mounted
    read-only under /kaggle/input). Without this step the resume logic finds
    nothing and retrains the full plan - 228 runs (107 folds x 2 models plus
    the unweighted controls), ~4.4 h on a T4 - which defeats the purpose of a
    targeted re-run.

    Only runs when out_dir has no store yet, so it can never overwrite work in
    progress.
    """
    dest = os.path.join(cfg["out_dir"], "predictions_finetuned")
    if any(os.path.exists(dest + ext) for ext in (".parquet", ".csv")):
        log.info("Store already present in %s; not seeding.", cfg["out_dir"])
        return False

    hits = sorted(glob.glob("/kaggle/input/**/predictions_finetuned.parquet",
                            recursive=True))
    hits += sorted(glob.glob("/kaggle/input/**/predictions_finetuned.csv",
                             recursive=True))
    if not hits:
        log.info("No previous store under /kaggle/input - starting from scratch.")
        return False

    src = hits[0]
    df = read_table(str(Path(src).with_suffix("")))
    if df is None or "fold" not in df.columns:
        log.warning("Found %s but could not read it; starting from scratch.", src)
        return False

    write_table(df, dest)
    log.info("SEEDED %d rows from a previous run (%s). Folds already complete "
             "will be skipped; families in force_rerun_families will be purged "
             "and retrained.", len(df), src)
    return True


def conform_store(df, cfg):
    """Bring a store written under an older schema up to STORE_COLS.

    Columns added later are derived where the information is already present
    (cost_usd from latency) and filled with the constant that was implicitly
    true at the time otherwise (quantization: the encoders were never
    quantised). Column ORDER is normalised too, because Stage 3's verification
    gate compares `list(store.columns) == STORE_COLS` exactly.
    """
    df = df.copy()
    if "train_epochs" not in df.columns:
        df["train_epochs"] = np.nan
    if "cost_usd" not in df.columns:
        df["cost_usd"] = (pd.to_numeric(df.get("latency_s"), errors="coerce")
                          .fillna(0.0) / 3600.0 * cfg["gpu_usd_per_hour"]).round(10)
    if "quantization" not in df.columns:
        df["quantization"] = "none_fp32"
    if "timestamp" not in df.columns:
        df["timestamp"] = ""
    missing = [c for c in STORE_COLS if c not in df.columns]
    for c in missing:
        df[c] = np.nan
    if missing:
        log.warning("Back-filled %d missing column(s) on the existing store: %s",
                    len(missing), missing)
    return df[STORE_COLS]


def run(uni, uni_by_id, splits, cfg):
    store_path = os.path.join(cfg["out_dir"], "predictions_finetuned")
    existing = read_table(store_path)

    # Fold sizes for every family, resumed rows included - see FOLD_TRAIN_N.
    for _fam, _entries in splits.items():
        for _e in _entries:
            FOLD_TRAIN_N[(_fam, str(_e["fold"]))] = len(_e["train_ids"])

    # Back-fill any column added after an older store was written, so a resume
    # never fails on a missing key and never silently drops a column. cost_usd
    # is DERIVED from the stored latency rather than left NaN, so a store
    # written before the canonical schema still reconciles exactly with a fresh
    # one instead of arriving at Stage 5 as zeros.
    if existing is not None:
        existing = conform_store(existing, cfg)

    existing, n_purged = purge_families(existing, cfg)

    # Resumed rows must describe the SAME folds splits.json now declares.
    # StratifiedGroupKFold's assignment is scikit-learn-version-dependent: a
    # Stage 1 re-run under a different sklearn regenerates cross-project folds
    # whose names match the old ones but whose contents differ, and the
    # (model_tag, fold) resume key would then silently mix predictions from
    # two different partitions. Comparing stored ids per fold against
    # splits.json catches that class of corruption outright.
    if existing is not None and len(existing):
        _split_ids = {(str(fam), str(e["fold"])): set(map(str, e["test_ids"]))
                      for fam, v in splits.items() for e in v}
        _mism = []
        for (_fam, _fold), _grp in existing.groupby(["family", "fold"]):
            want = _split_ids.get((str(_fam), str(_fold)))
            if want is None:
                _mism.append((_fam, _fold, "fold missing from splits.json"))
            elif set(_grp["id"].astype(str)) != want:
                _mism.append((_fam, _fold,
                              f"{_grp['id'].nunique()} stored ids vs "
                              f"{len(want)} in splits.json"))
        if _mism:
            raise ValueError(
                f"Resumed store does not match splits.json fold contents "
                f"(first mismatches: {_mism[:5]}). This happens when Stage 1 "
                f"was re-run under a different scikit-learn version - fold "
                f"NAMES match but their CONTENTS differ. Attach the SAME "
                f"committed Stage 1 output this store was built from, or "
                f"delete predictions_finetuned.* and retrain from scratch.")

    # A resumed store can carry rows trained under an OLDER epoch plan, and
    # the plan-level guard below cannot see them: it validates only what THIS
    # run would train. Rows trained at a different budget than the current
    # task_epochs/family_epochs would silently mix into every downstream
    # table, which is exactly the confound the guard exists to prevent - so
    # check the stored train_epochs against the current plan and stop with an
    # actionable message instead.
    if existing is not None and len(existing) and not cfg["quick_mode"]:
        _stale = []
        _epochs_num = pd.to_numeric(existing["train_epochs"], errors="coerce")
        for (_fam, _task), _grp in existing.assign(_ep=_epochs_num).groupby(
                ["family", "task"]):
            _planned = epochs_for(_fam, _task, cfg)
            _got = sorted(set(int(x) for x in _grp["_ep"].dropna().unique()))
            _wrong = [x for x in _got if x != _planned]
            if _wrong:
                _stale.append((_fam, _task, _wrong, _planned))
        if _stale:
            raise ValueError(
                f"Resumed rows were trained at epoch counts that differ from "
                f"the current plan: {_stale} (family, task, stored, planned). "
                f"Add those families to force_rerun_families so they are "
                f"purged and retrained, or align task_epochs/family_epochs "
                f"with the store.")

    records = existing.to_dict("records") if existing is not None else []
    done = set(zip(existing.model_tag, existing.fold)) if existing is not None else set()
    if done:
        log.info("Resuming: %d rows kept, %d (model,fold) already complete%s.",
                 len(records), len(done),
                 f", {n_purged} rows purged for retraining" if n_purged else "")

    q = cfg["quick_mode"]
    base_hp = {"max_len": cfg["max_len"], "lr": cfg["lr"],
               "train_bs": cfg["train_bs"], "eval_bs": cfg["eval_bs"]}

    plan = build_plan(splits, cfg)
    todo = [p for p in plan if (p[0], p[4]["fold"]) not in done]
    log.info("Plan: %d runs total, %d remaining.", len(plan), len(todo))

    ep_plan = {}
    for _, _, _, fam, e in plan:
        ep_plan[fam] = epochs_for(fam, e["task"], cfg)
    log.info("Epochs per family: %s", ep_plan)
    # Same task, two training budgets, is the confound task_epochs exists to
    # prevent. Check it against the plan rather than trusting the config to be
    # read correctly, and stop before spending GPU hours on an unusable gap.
    by_task = {}
    for _, _, _, fam, e in plan:
        by_task.setdefault(e["task"], {}).setdefault(
            epochs_for(fam, e["task"], cfg), []).append(fam)
    split_tasks = {t: {ep: sorted(set(f)) for ep, f in d.items()}
                   for t, d in by_task.items() if len(d) > 1}
    if split_tasks:
        raise ValueError(
            f"Task(s) planned at more than one epoch count: {split_tasks}. "
            f"A generalisation gap computed across those arms would subtract "
            f"models trained on different budgets. Fix task_epochs / "
            f"family_epochs before running.")

    t_start = time.time()
    for i, (tag, base, weighting, family, e) in enumerate(todo, 1):
        labelset = e.get("labelset") or BINARY_LABELSETS[e["task"]]
        tr = uni_by_id.loc[e["train_ids"]].reset_index()
        te = uni_by_id.loc[e["test_ids"]].reset_index()

        n_epochs = epochs_for(family, e["task"], cfg)
        FOLD_TRAIN_N[(family, str(e["fold"]))] = len(e["train_ids"])
        hp = dict(base_hp, epochs=n_epochs)

        pred_df, train_time = finetune_and_predict(
            base, tr, te, e["label_col"], labelset, hp, weighting)

        pred_df = pred_df.assign(
            model_tag=tag, model_type="finetuned", approach="finetuned",
            class_weighting=weighting,
            task=e["task"], dataset=dataset_of(uni_by_id, e["test_ids"]),
            eval_regime=regime_of(family, e), family=family, fold=e["fold"],
            split="supervised", prompt_id="supervised", shot_k=-1,
            parse_ok=True, train_time_s=round(train_time, 3),
            train_epochs=n_epochs,
            price_in_per_mtok=0.0, price_out_per_mtok=0.0,
            # Same cost model Stages 3/3b use for local compute: wall-clock
            # inference seconds x the imputed hourly rate. Storing it here means
            # Stage 5 reads one column for every tier instead of special-casing
            # the encoder, and the aggregate can never drift from the rows.
            cost_usd=(pred_df["latency_s"] / 3600.0 * cfg["gpu_usd_per_hour"]).round(10),
            cost_basis="gpu_seconds",
            quantization="none_fp32",
            timestamp=time.strftime("%Y-%m-%dT%H:%M:%S"),
            raw_output="")
        records.extend(pred_df[STORE_COLS].to_dict("records"))
        if i % int(cfg.get("checkpoint_every", 5)) == 0:      # PERF-01
            write_table(pd.DataFrame(records, columns=STORE_COLS), store_path)

        elapsed = time.time() - t_start
        eta = elapsed / i * (len(todo) - i)
        log.info("[%3d/%3d] %-26s | %-15s | %-28s train=%4d test=%4d | ep=%2d | "
                 "%5.1fs | ETA %4.1f min",
                 i, len(todo), tag, e["eval_regime"], e["fold"],
                 len(tr), len(te), n_epochs, train_time, eta / 60)

    out = pd.DataFrame(records, columns=STORE_COLS)
    write_table(out, store_path)                              # final flush
    log.info("Done. Prediction store -> %s.parquet/.csv (%d rows).",
             store_path, len(records))
    return out


# =============================================================================
# summaries
# =============================================================================
def summarise(store):
    """CV families are POOLED: every requirement is tested exactly once, so
    pooling gives one honest macro-F1 rather than an average of noisy per-fold
    scores. Cross-dataset directions are reported separately because they are
    substantively different experiments."""
    rows = []
    keys = ["model_tag", "class_weighting", "task", "dataset", "eval_regime", "family"]
    for kv, g in store.groupby(keys, dropna=False):
        rec = dict(zip(keys, kv))
        # FIXED label set, not the observed one. A fold that never saw
        # 'portability' must still divide its macro average by 11, or this table
        # reports a higher macro-F1 than Stage 4's tab1 for the very same cell.
        labelset = LABELSETS.get(str(rec["task"]))
        if labelset is None:
            labelset = sorted(set(g.y_true.astype(str)) | set(g.y_pred.astype(str)))
            log.warning("No declared label set for task %r; falling back to the "
                        "observed labels. This cell may disagree with Stage 4.",
                        rec["task"])
        rec["n_folds"] = g["fold"].nunique()
        rec.update(compute_metrics(g.y_true.astype(str), g.y_pred.astype(str), labelset))
        rec["mean_latency_s"] = round(float(g.latency_s.mean()), 6)
        # A gap = in_domain - transfer is only a regime effect if the two arms
        # trained on comparable amounts of data. The epoch guard covers epochs;
        # nothing covered SIZE. in_domain_security_secreq trains on ~355 items
        # per fold while its cross-project counterpart trains on ~296, and that
        # difference is currently invisible in every table. Recording it here
        # means the asymmetry can be quoted, or corrected, rather than missed.
        _sizes = [FOLD_TRAIN_N[(str(rec["family"]), f)]
                  for f in g["fold"].astype(str).unique()
                  if (str(rec["family"]), f) in FOLD_TRAIN_N]
        rec["mean_train_n"] = round(float(np.mean(_sizes)), 1) if _sizes else np.nan
        rec["total_train_time_s"] = round(
            float(g.groupby("fold")["train_time_s"].first().sum()), 2)
        # Surfaced so the deviation is visible in the results table itself and
        # cannot be forgotten when writing the methodology section.
        eps = g["train_epochs"].dropna().unique() if "train_epochs" in g else []
        rec["train_epochs"] = int(eps[0]) if len(eps) == 1 else (
            "MIXED:" + ",".join(str(int(x)) for x in sorted(eps)) if len(eps) else "unknown")
        rows.append(rec)
    out = pd.DataFrame(rows)
    return out.sort_values(["task", "dataset", "eval_regime", "macro_f1"],
                           ascending=[True, True, True, False])


def weighting_effect(summary):
    """The class-imbalance claim, with real numbers on BOTH sides of the gap."""
    d = summary[summary["class_weighting"].isin(["balanced", "none"])].copy()
    d["base"] = d["model_tag"].str.replace("-weighted|-unweighted", "", regex=True)
    piv = d.pivot_table(index=["base", "task", "dataset", "eval_regime"],
                        columns="class_weighting",
                        values=["macro_f1", "weighted_f1", "macro_recall"])
    piv = piv.dropna(how="any")

    # The unweighted control may legitimately be absent: quick_mode skips it by
    # design, and a resumed run may not have reached it yet. Return an empty
    # frame rather than raising a KeyError, so a rehearsal run still completes
    # and writes its summary.
    have = set(piv.columns.get_level_values(1))
    if not {"balanced", "none"} <= have:
        log.warning("Class-weighting control not present (found: %s). "
                    "Skipping the weighting-effect table. This is expected in "
                    "quick_mode; in the full run it means the control folds "
                    "have not finished.", sorted(have))
        return pd.DataFrame()

    if len(piv):
        piv[("macro_f1", "delta")] = piv[("macro_f1", "balanced")] - piv[("macro_f1", "none")]
    return piv


def generalisation_gap(summary):
    """RQ2, computed as an actual difference.

    DIAGNOSTIC ONLY. Stage 4's tab3_generalisation_gap.csv is the authoritative
    table for the write-up: it applies the evaluability gate (MIN_CLASS_SUPPORT,
    MIN_N) and separates cross_project from cross_dataset instead of taking the
    first match. This function shares Stage 4's label sets so the two agree
    numerically, but quote tab3. Two tables describing one quantity is how a
    thesis ends up with two different numbers for its headline claim."""
    rows = []
    for (tag, task, dataset), g in summary[
            summary.class_weighting == "balanced"].groupby(
            ["model_tag", "task", "dataset"]):
        ind = g[g.eval_regime == "in_domain"]
        if not len(ind):
            continue
        base = float(ind.iloc[0].macro_f1)
        base_n = ind.iloc[0].get("mean_train_n", np.nan)
        for regime in ("cross_project", "cross_dataset"):
            oos = g[g.eval_regime == regime]
            if not len(oos):
                continue
            # A gap is a REGIME effect only if the two arms trained on
            # comparable amounts of data. The epoch guard enforces one training
            # budget per task; nothing enforced size, and it is unmatched in two
            # of these cells (security/SECREQ: ~355 in-domain against ~296
            # cross-project). Reported beside the gap so it is quoted or
            # corrected, never missed.
            oos_n = oos.iloc[0].get("mean_train_n", np.nan)
            ratio = (round(float(oos_n) / float(base_n), 3)
                     if base_n and oos_n and float(base_n) else np.nan)
            rows.append({
                "model_tag": tag, "task": task, "dataset": dataset,
                "in_domain_macro_f1": round(base, 4),
                "oos_regime": regime,
                "oos_macro_f1": round(float(oos.iloc[0].macro_f1), 4),
                "in_domain_mean_train_n": base_n,
                "oos_mean_train_n": oos_n,
                "train_size_ratio": ratio,
                # NaN when the ratio is unknown - False would claim a measured
                # mismatch that was never measured.
                "size_matched": (bool(0.9 <= ratio <= 1.1)
                                 if ratio == ratio else np.nan),
                "generalisation_gap": round(base - float(oos.iloc[0].macro_f1), 4),
                "relative_drop_pct": round(
                    100 * (base - float(oos.iloc[0].macro_f1)) / base, 1) if base else np.nan,
            })
    return pd.DataFrame(rows).sort_values(["task", "dataset", "model_tag"])


if __name__ == "__main__":
    CONFIG["data_dir"] = find_data_dir(CONFIG["data_dir"])
    log.info("Stage 1 artefacts: %s", CONFIG["data_dir"])
    uni = read_table(str(Path(CONFIG["data_dir"]) / "unified"))
    with open(Path(CONFIG["data_dir"]) / "splits.json") as f:
        splits = json.load(f)
    uni_by_id = uni.set_index("id")
    log.info("Loaded unified=%d rows | split families=%d", len(uni), len(splits))

    if CONFIG["families"] is None:
        CONFIG["families"] = [f for f in splits
                              if f not in set(CONFIG.get("skip_families", []))]
        log.info("Running every family Stage 1 emits: %d families, %d folds.",
                 len(CONFIG["families"]),
                 sum(len(splits[f]) for f in CONFIG["families"]))
    missing = [f for f in CONFIG["families"] if f not in splits]
    if missing:
        raise KeyError(f"splits.json is missing {missing}. Re-run Stage 1 -- you "
                       f"are using an old splits.json without those arms.")

    # Report the (task x regime) coverage this run will actually produce, so a
    # gap is visible here rather than discovered when a Stage 4 table is empty.
    _cov = {}
    for f in CONFIG["families"]:
        for e in splits[f]:
            _cov[(e["task"], e["eval_regime"])] = _cov.get(
                (e["task"], e["eval_regime"]), 0) + 1
    _tasks = sorted({t for t, _ in _cov})
    _regs = ["in_domain", "cross_project", "cross_dataset"]
    log.info("RQ2 coverage for this run:")
    for t in _tasks:
        log.info("    %-14s %s", t, "  ".join(
            f"{r}={_cov.get((t, r), 0)}" for r in _regs))
    _gaps = [f"{t}/{r}" for t in _tasks for r in _regs if not _cov.get((t, r))]
    if _gaps:
        log.warning("UNCOVERED (task, regime) cells: %s. These are data limits, "
                    "not bugs - report them rather than claiming coverage.",
                    ", ".join(_gaps))

    Path(CONFIG["out_dir"]).mkdir(parents=True, exist_ok=True)
    seed_store_from_inputs(CONFIG)
    store = run(uni, uni_by_id, splits, CONFIG)

    summary = summarise(store)
    summary.to_csv(os.path.join(CONFIG["out_dir"], "summary_finetuned.csv"), index=False)

    # Integrity guard: a family trained partly at 3 epochs and partly at 10 is a
    # silent confound. Fail loudly rather than let it into the results.
    mixed = [str(v) for v in summary["train_epochs"] if str(v).startswith("MIXED")]
    if mixed:
        log.error("MIXED EPOCH SETTINGS DETECTED within a result group: %s. "
                  "Add the affected families to force_rerun_families and re-run, "
                  "or delete predictions_finetuned.* and start clean.", set(mixed))
    unknown = int((summary["train_epochs"] == "unknown").sum())
    if unknown:
        log.warning("%d group(s) have no recorded epoch count (rows predate the "
                    "train_epochs column). Harmless, but note it if you report "
                    "hyperparameters per family.", unknown)

    we = weighting_effect(summary)
    if len(we):
        we.to_csv(os.path.join(CONFIG["out_dir"], "class_weighting_effect.csv"))

    gap = generalisation_gap(summary)
    # Named as a diagnostic so it can never be mistaken for Stage 4's tab3,
    # which is the table the thesis quotes.
    gap.to_csv(os.path.join(CONFIG["out_dir"],
                            "finetuned_generalisation_gap_DIAGNOSTIC.csv"),
               index=False)

    with open(os.path.join(CONFIG["out_dir"], "stage2_manifest.json"), "w") as f:
        import transformers
        json.dump({"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "seed": SEED,
                   "device": DEVICE, "python": platform.python_version(),
                   "torch": torch.__version__,
                   "transformers": transformers.__version__,
                   "config": CONFIG}, f, indent=2, default=str)

    pd.set_option("display.width", 220); pd.set_option("display.max_columns", None)
    print("\n" + "=" * 92)
    print("STAGE 2 SUMMARY - fine-tuned baselines")
    print("=" * 92)
    print(summary.to_string(index=False))

    print("\n" + "=" * 92)
    print("EFFECT OF CLASS WEIGHTING  (balanced vs none)")
    print("A macro-recall near 1/n_classes means the model collapsed to one class.")
    print("=" * 92)
    print(we.to_string() if len(we) else "  (control not run)")

    print("\n" + "=" * 92)
    print("RQ2 - GENERALISATION GAP  (in-domain minus out-of-distribution)")
    print("DIAGNOSTIC. Quote Stage 4's tab3_generalisation_gap.csv, which applies")
    print("the evaluability gate and keeps cross_project separate from cross_dataset.")
    print("=" * 92)
    print(gap.to_string(index=False) if len(gap) else "  (no in-domain arm found)")
    print("\nNext: Stage 3 (LLM harness) appends to this identical schema.")
    print("=" * 92)

# =============================================================================
# CLASS-WEIGHTING AUDIT NOTE  (read before writing the thesis section)
#
# Implementation status, stated precisely so the thesis can match it:
#   * Weighting is inverse-frequency ("balanced", sklearn compute_class_weight)
#     applied as the `weight=` argument of F.cross_entropy. It is NOT resampling
#     and NOT SMOTE. If Chapter 2 cites SMOTE as the field's usual remedy, say
#     explicitly that this study uses loss weighting instead, and why: SMOTE
#     synthesises text-space neighbours, which is ill-defined for requirement
#     sentences.
#   * The primary condition is weighted for EVERY model, task and fold.
#   * The unweighted control runs on the families in CONFIG["control_families"].
#
# On the evidence from the previous run, weighting changed macro-F1 by at most
# ~0.018 and helped in only one of four cross-dataset comparisons; macro-recall
# never approached the collapse threshold. If that reproduces here, do NOT write
# that weighting "fixes a class-imbalance collapse". Write instead that the
# control confirms the effect is small and inconsistent in sign, and that the
# cross-dataset gap is therefore attributable to distribution shift rather than
# to imbalance handling. That is a stronger claim, and it is the one the data
# supports.
# =============================================================================
